<a href="https://colab.research.google.com/github/JuliannaLGo/Final-Project-CS2/blob/main/4Q_FA5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import json
url = "https://raw.githubusercontent.com/JuliannaLGo/Final-Project-CS2/refs/heads/main/data/library.json"
response = requests.get(url)
library = response.json()
print(library)

[{'borrower_id': 1, 'student': 'Carlos Mendoza', 'books': [{'book_title': 'Introduction to Python', 'author': 'Guido van Rossum', 'borrow_date': '2025-08-20', 'return_date': '2025-09-01'}, {'book_title': 'Advanced Algebra', 'author': 'Richard Brown', 'borrow_date': '2025-09-10', 'return_date': '2025-09-20'}, {'book_title': 'World History Volume 1', 'author': 'Arthur Kemp', 'borrow_date': '2025-09-15', 'return_date': None}]}, {'borrower_id': 2, 'student': 'Ella Cruz', 'books': [{'book_title': 'Data Science for Beginners', 'author': 'Jane Smith', 'borrow_date': '2025-09-05', 'return_date': '2025-09-12'}, {'book_title': 'Creative Writing Essentials', 'author': 'Maria Johnson', 'borrow_date': '2025-09-10', 'return_date': None}, {'book_title': 'Physics Fundamentals', 'author': 'Isaac Newton', 'borrow_date': '2025-09-18', 'return_date': None}]}, {'borrower_id': 3, 'student': 'Mark Dela Peña', 'books': [{'book_title': 'The Great Gatsby', 'author': 'F. Scott Fitzgerald', 'borrow_date': '2025-0

In [2]:
!pip install firebase-admin

In [3]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase_key.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://jasmine-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"})
print("Firebase connected successfully!")


Firebase connected successfully!


In [4]:
import requests
import json

data = library
print("JSON file loaded")
ref = db.reference("library")
for item in data:
  ref.child(str(item["borrower_id"])).set(item)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [12]:
ref = db.reference("library")
while True:
    print("\n===== STUDENT DATABASE MENU =====")
    print("1. Display Borrower Data")
    print("2. Add Borrower")
    print("3. Update Borrower")
    print("4. Delete Borrower")
    print("5. Features")
    print("6. Exit")

    choice = input("Enter choice: ")


 # DISPLAY
    if choice == "1":
        print("\nLibrary Records:")
        current_library_data = ref.get()
        if current_library_data:
            borrowers = []
            # Handle cases where Firebase might return a dictionary or a list
            if isinstance(current_library_data, dict):
                for value in current_library_data.values():
                    if value: # Ensure value is not None
                        borrowers.append(value)
            elif isinstance(current_library_data, list):
                for value in current_library_data:
                    if value: # Ensure value is not None (e.g., index 0 can be None)
                        borrowers.append(value)

            if borrowers: # Only proceed if there are actual borrower records after filtering
                print("\n" + "="*80)
                print(f"{'Borrower ID':<15} {'Student Name':<30}")
                print("="*80)
                for borrower in borrowers:
                    if borrower:
                        print(f"{borrower['borrower_id']:<15} {borrower['student']:<30}")
                        if 'books' in borrower and borrower['books']:
                            print("  " + "-"*78)
                            print(f"  {'Book Title':<30} {'Author':<25} {'Borrowed':<12} {'Returned':<12}")
                            print("  " + "-"*78)
                            for book in borrower["books"]:
                                return_status = book.get('return_date') if book.get('return_date') else 'Not Returned'
                                print(f"  {book['book_title']:<30} {book['author']:<25} {book['borrow_date']:<12} {return_status:<12}")
                            print("  " + "-"*78)
                        else:
                            print("  No books borrowed.")
                        print("="*80) # Separator between borrowers
            else:
                print("No borrower records found.") # This covers cases where current_library_data was [None, None]
        else:
            print("No borrower records found.")

 # ADD
    elif choice == "2":
        sid = input("Enter ID: ")
        name = input("Enter name: ")
        new_books_list = []
        while True:
            print(f"\nEnter details for book:")
            book_title = input("Enter Book Name: ")
            author = input("Enter Author Name: ")
            borrow_date = input("Enter Borrow Date (YYYY-MM-DD): ")
            return_date = input("Enter Return Date (YYYY-MM-DD, or leave empty if not returned): ")
            new_books_list.append({
                "book_title": book_title,
                "author": author,
                "borrow_date": borrow_date,
                "return_date": return_date if return_date else None

            })
            x = input("Do you want to add another book? (y/n): ")
            if x.lower() != "y":
                break

        new_borrower_entry = {
            "borrower_id": int(sid),
            "student": name,
            "books": new_books_list
        }


        ref.child(str(sid)).set(new_borrower_entry)
        print("Borrower added successfully!")

    # UPDATE
    elif choice == "3":
        sid = input("Enter ID of Borrower to update: ")
        borrower_ref = db.reference("library").child(str(sid)) # Ensure using str(sid) for child key
        borrower_data = borrower_ref.get()

        if borrower_data:
            print(f"\nCurrently updating Borrower ID: {borrower_data['borrower_id']}, Name: {borrower_data['student']}")


            new_name = input(f"Enter new name for borrower (current: {borrower_data['student']}): ")
            if not new_name: # If user leaves blank, keep old name
                new_name = borrower_data['student']


            current_books_info = ""
            if 'books' in borrower_data and borrower_data['books']:
                current_books_info = ", ".join([book['book_title'] for book in borrower_data['books']])
                print(f"Current books: {current_books_info}")
            else:
                print("Current books: None")

            new_books_list = []
            print("\n--- Enter NEW Book Details (will replace existing books) ---")
            while True:
                add_more = input("Do you want to add a book to this borrower? (y/n): ")
                if add_more.lower() != "y":
                    break

                print(f"\nEnter details for book {len(new_books_list) + 1}:")
                book_title = input("Enter Book Name: ")
                author = input("Enter Author Name: ")
                borrow_date = input("Enter Borrow Date (YYYY-MM-DD): ")
                return_date = input("Enter Return Date (YYYY-MM-DD, or leave empty if not returned): ")
                new_books_list.append({
                    "book_title": book_title,
                    "author": author,
                    "borrow_date": borrow_date,
                    "return_date": return_date if return_date else None
                })

            updated_borrower_entry = {
                "borrower_id": int(sid), # Keep ID consistent
                "student": new_name,
                "books": new_books_list
            }

            borrower_ref.set(updated_borrower_entry) # Use set to overwrite the entire entry
            print("Borrower updated successfully!")
        else:
            print("Borrower not found.")

    # DELETE
    elif choice == "4":
        sid = input("Enter Borrower ID to delete: ")
        borrower_to_delete_ref = ref.child(str(sid))

        if borrower_to_delete_ref.get():
            borrower_to_delete_ref.delete()
            print("Borrower deleted successfully!")
        else:
            print("Borrower not found.")

    # FEATURES
    elif choice == "5":
        print("\n===== LIBRARY DATABASE FEATURES =====")
        print("1. Search Book")
        print("2. Display Borrow History")
        print("3. Feature 3")
        print("4. Feature 4")
        print("5. Feature 5")
        print("6. Feature 6")
        print("7. Back to Main Menu")
        choice = input("Enter choice: ")

        match choice:
          case "1":
            library_data = ref.get()
            search_term = input("Enter book title to search: ").lower()
            found_books = []

            if library_data:
                if isinstance(library_data, dict):
                    all_borrowers_data = library_data.values()
                elif isinstance(library_data, list):
                    all_borrowers_data = [b for b in library_data if b is not None]
                else:
                    all_borrowers_data = []

                for borrower in all_borrowers_data:
                    if 'books' in borrower and borrower['books']:
                        for book in borrower['books']:
                            if search_term in book['book_title'].lower():
                                found_books.append({
                                    'borrower_id': borrower['borrower_id'],
                                    'student': borrower['student'],
                                    'book_title': book['book_title'],
                                    'author': book['author'],
                                    'borrow_date': book['borrow_date'],
                                    'return_date': book.get('return_date', 'Not Returned')
                                })

            if found_books:
                print(f"\nFound {len(found_books)} book(s) matching '{search_term}':")
                print("\n" + "="*90)
                print(f"{'Borrower ID':<15} {'Student Name':<25} {'Book Title':<30} {'Status':<15}")
                print("="*90)
                for book_info in found_books:
                    status = 'Returned' if book_info['return_date'] != 'Not Returned' else 'Not Returned'
                    print(f"{book_info['borrower_id']:<15} {book_info['student']:<25} {book_info['book_title']:<30} {status:<15}")
                print("="*90)
            else:
                print(f"No books found with title containing '{search_term}'.")

          case "2":
            borrower_id = input("Enter borrower ID: ")
            borrower_data = ref.child(borrower_id).get()

            if borrower_data:
                print(f"\nBorrow History for: {borrower_data['student']} (ID: {borrower_data['borrower_id']})")
                if 'books' in borrower_data and borrower_data['books']:
                    print("  " + "-"*78)
                    print(f"  {'Book Title':<30} {'Author':<25} {'Borrowed':<12} {'Returned':<12}")
                    print("  " + "-"*78)
                    for book in borrower_data['books']:
                        return_status = book.get('return_date') if book.get('return_date') else 'Not Returned'
                        print(f"  {book['book_title']:<30} {book['author']:<25} {book['borrow_date']:<12} {return_status:<12}")
                    print("  " + "-"*78)
                else:
                    print("  No books borrowed by this student.")
            else:
                print("Borrower not found.")

          case "7":
            print("Returning to Main Menu...")

          case _:
            print("Invalid choice. Try again.")


    # EXIT
    elif choice == "6":
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Try again.")


===== STUDENT DATABASE MENU =====
1. Display Borrower Data
2. Add Borrower
3. Update Borrower
4. Delete Borrower
5. Features
6. Exit
Enter choice: 5

===== LIBRARY DATABASE FEATURES =====
1. Search Book
2. Display Borrow History
3. Borrower Book Count
4. Unreturned Books Per Borrower
5. Books By Author
6. Unreturned Books
7. Back to Main Menu
Enter choice: 7
Returning to Main Menu...

===== STUDENT DATABASE MENU =====
1. Display Borrower Data
2. Add Borrower
3. Update Borrower
4. Delete Borrower
5. Features
6. Exit
Enter choice: 6
Exiting program...
